In [1]:
# Quick smoke-test training (1 epoch) — paste into a new notebook cell and run
import json
from pathlib import Path
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, optimizers
import tensorflow as tf
import os

BASE = Path(r"C:\MedicinalPlantApp")
DATA_DIR = BASE / "data"
MODELS_DIR = BASE / "models"
MODELS_DIR.mkdir(exist_ok=True)

IMG_SIZE = 160        # smaller for speed; you can use 224 later
BATCH = 16            # reduce if you run out of memory
EPOCHS = 1            # smoke test

# generators
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=20,
                                   width_shift_range=0.1,
                                   height_shift_range=0.1,
                                   zoom_range=0.1,
                                   horizontal_flip=True)
val_datagen = ImageDataGenerator(rescale=1./255)

train_dir = str(DATA_DIR / "train")
val_dir = str(DATA_DIR / "val")

train_flow = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode='categorical'
)
val_flow = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode='categorical',
    shuffle=False
)

# save class indices for later inference
with open(MODELS_DIR / "class_indices.json", "w") as f:
    json.dump(train_flow.class_indices, f)
print("Class indices saved:", train_flow.class_indices)

num_classes = len(train_flow.class_indices)
print("Num classes:", num_classes)

# build model (transfer learning)
base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(IMG_SIZE,IMG_SIZE,3))
base.trainable = False
x = base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation="relu")(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)
model = models.Model(inputs=base.input, outputs=outputs)

model.compile(optimizer=optimizers.Adam(learning_rate=1e-3),
              loss="categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

# callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(str(MODELS_DIR / "plant_model.h5"), save_best_only=True)
]

# train (1 epoch smoke-test)
history = model.fit(train_flow, validation_data=val_flow, epochs=EPOCHS, callbacks=callbacks)

# print short report
print("Finished training. History keys:", history.history.keys())
print("Last epoch - train acc:", history.history.get("accuracy")[-1], "val acc:", history.history.get("val_accuracy")[-1] if "val_accuracy" in history.history else None)

# ensure model file exists
print("Saved model exists:", (MODELS_DIR / "plant_model.h5").exists())


Found 11504 images belonging to 31 classes.
Found 2876 images belonging to 31 classes.
Class indices saved: {'.ipynb_checkpoints': 0, 'Arive-Dantu': 1, 'Basale': 2, 'Betel': 3, 'Crape_Jasmine': 4, 'Curry': 5, 'Drumstick': 6, 'Fenugreek': 7, 'Guava': 8, 'Hibiscus': 9, 'Indian_Beech': 10, 'Indian_Mustard': 11, 'Jackfruit': 12, 'Jamaica_Cherry-Gasagase': 13, 'Jamun': 14, 'Jasmine': 15, 'Karanda': 16, 'Lemon': 17, 'Mango': 18, 'Mexican_Mint': 19, 'Mint': 20, 'Neem': 21, 'Oleander': 22, 'Parijata': 23, 'Peepal': 24, 'Pomegranate': 25, 'Rasna': 26, 'Rose_apple': 27, 'Roxburgh_fig': 28, 'Sandalwood': 29, 'Tulsi': 30}
Num classes: 31
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 8s 1us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 160, 160, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1 (Conv2D)                │ (None, 80, 80, 32)        │             864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_Conv1 (BatchNormalization) │ (None, 80, 80, 32)        │             128 │ Conv1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1_relu (ReLU)             │ (None, 80, 80, 32)        │               0 │ bn_Conv1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 80, 80, 32)        │             288 │ Conv1_relu[0][0]           │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_BN    │ (None, 80, 80, 32)        │             128 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_relu  │ (None, 80, 80, 32)        │               0 │ expanded_conv_depthwise_B… │
│ (ReLU)                        │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 80, 80, 16)        │             512 │ expanded_conv_depthwise_r… │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_BN      │ (None, 80, 80, 16)        │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand (Conv2D)       │ (None, 80, 80, 96)        │           1,536 │ expanded_conv_project_BN[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_BN             │ (None, 80, 80, 96)        │             384 │ block_1_expand[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_relu (ReLU)    │ (None, 80, 80, 96)        │               0 │ block_1_expand_BN[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_pad (ZeroPadding2D)   │ (None, 81, 81, 96)        │               0 │ block_1_expand_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_depthwise             │ (None, 40, 40, 96)        │             864 │ block_1_pad[0][0]          │
│ (DepthwiseConv2D)             │                           │               

 Total params: 2,593,887 (9.89 MB)

 Trainable params: 335,903 (1.28 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

C:\Users\shrey\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


719/719 ━━━━━━━━━━━━━━━━━━━━ 0s 535ms/step - accuracy: 0.7180 - loss: 0.9872

719/719 ━━━━━━━━━━━━━━━━━━━━ 465s 636ms/step - accuracy: 0.8396 - loss: 0.5134 - val_accuracy: 0.9392 - val_loss: 0.1870
Finished training. History keys: dict_keys(['accuracy', 'loss', 'val_accuracy', 'val_loss'])
Last epoch - train acc: 0.8396210074424744 val acc: 0.9391515851020813
Saved model exists: True
